# Model Validation

In [1]:
# ── §1 · Setup ────────────────────────────────────────────────────────────────
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, accuracy_score

sys.path.append('..')
from src.elo import load_nfl_data

from src.model import (
    build_features,
    walk_forward_validation,
    train_final_model,
    FEATURES,
)

---

## Construcción y validación de modelos

In [2]:
# Parámetros ELO calibrados en Notebook 02
HOME_ADVANTAGE = 42.2

# ── §2 · Cargar datos ─────────────────────────────────────────────────────────
df = load_nfl_data(range(1999, 2026))
history_df = pd.read_csv('../data/processed/elo_history.csv') 

print(f"Schedules  : {len(df):,} partidos")
print(f"ELO history: {len(history_df):,} partidos")

Schedules  : 7,276 partidos
ELO history: 7,276 partidos


In [3]:
# ── §3 · Feature engineering ──────────────────────────────────────────────────
features_df = build_features(history_df, df, home_advantage=HOME_ADVANTAGE)

print(f"\nDataset final: {len(features_df):,} partidos REG")
print(f"Seasons      : {features_df['season'].min()}–{features_df['season'].max()}")
print(f"\nEstadísticas de features:")
print(features_df[FEATURES].describe().round(3))


Dataset final: 6,967 partidos REG
Seasons      : 1999–2025

Estadísticas de features:
       elo_diff  home_rest  away_rest  div_game  week_norm
count  6967.000   6967.000   6967.000  6967.000   6967.000
mean     41.954      7.425      7.473     0.387      0.433
std     113.801      1.977      1.984     0.487      0.238
min    -317.472      4.000      4.000     0.000      0.045
25%     -32.678      7.000      7.000     0.000      0.227
50%      42.200      7.000      7.000     0.000      0.429
75%     118.356      7.000      7.000     1.000      0.636
max     491.547     16.000     21.000     1.000      0.818


In [4]:
# ── §4 · Walk-forward validation ──────────────────────────────────────────────
TEST_SEASONS = [2022, 2023, 2024, 2025]

print("Walk-forward validation:")
print("-" * 65)
results_df, metrics_df = walk_forward_validation(
    features_df,
    test_seasons=TEST_SEASONS,
)

print("\nMétricas por fold:")
print(metrics_df.to_string(index=False))

Walk-forward validation:
-----------------------------------------------------------------
Modelo          : LogisticRegression
Scale features  : True
Features        : ['elo_diff', 'home_rest', 'away_rest', 'div_game', 'week_norm']
-----------------------------------------------------------------
Fold 2022 | Train=5,880 | Test=271 | BS=0.2305 | Acc=61.6%
Fold 2023 | Train=6,151 | Test=272 | BS=0.2338 | Acc=61.4%
Fold 2024 | Train=6,423 | Test=272 | BS=0.2116 | Acc=67.3%
Fold 2025 | Train=6,695 | Test=272 | BS=0.2232 | Acc=67.3%

Métricas por fold:
 test_season  train_size  test_size  brier_score  log_loss  accuracy
        2022        5880        271       0.2305    0.6512    0.6162
        2023        6151        272       0.2338    0.6623    0.6140
        2024        6423        272       0.2116    0.6108    0.6728
        2025        6695        272       0.2232    0.6370    0.6728


In [5]:
# ── §5 · Métricas  ───────────────────────────────────────────────────
results_only_elo, _ = walk_forward_validation(
    features_df,
    test_seasons=TEST_SEASONS,
    feature_cols=['elo_diff'],   # ← solo el feature principal
)

bs_only_elo = brier_score_loss(
    results_only_elo['home_team_win'],
    results_only_elo['p_model']
)
print(f"BS logístico con solo elo_diff: {bs_only_elo:.4f}")

Modelo          : LogisticRegression
Scale features  : True
Features        : ['elo_diff']
-----------------------------------------------------------------
Fold 2022 | Train=5,880 | Test=271 | BS=0.2284 | Acc=63.1%
Fold 2023 | Train=6,151 | Test=272 | BS=0.2362 | Acc=60.7%
Fold 2024 | Train=6,423 | Test=272 | BS=0.2112 | Acc=65.4%
Fold 2025 | Train=6,695 | Test=272 | BS=0.2227 | Acc=66.2%
BS logístico con solo elo_diff: 0.2246


In [6]:
model, scaler, coef_df = train_final_model(features_df)
print(coef_df.to_string(index=False))

  feature  coefficient  abs_coef
 elo_diff       0.6790    0.6790
away_rest      -0.0570    0.0570
 div_game      -0.0544    0.0544
week_norm       0.0332    0.0332
home_rest       0.0153    0.0153


In [7]:
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


C_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
c_results = []

for C in C_values:

    all_preds = []

    for test_season in TEST_SEASONS:
        train = features_df[features_df['season'] <  test_season]
        test  = features_df[features_df['season'] == test_season]

        X_train = train[FEATURES]
        y_train = train['home_team_win']
        w_train = train['sample_weight']
        X_test  = test[FEATURES]
        y_test  = test['home_team_win']

        scaler     = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        model = LogisticRegression(C=C, max_iter=1000, random_state=42)
        model.fit(X_train_sc, y_train, sample_weight=w_train)

        p_home = model.predict_proba(X_test_sc)[:, 1]
        all_preds.append(pd.DataFrame({
            'y_true' : y_test.values,
            'p_model': p_home
        }))

    all_preds_df = pd.concat(all_preds)
    bs  = brier_score_loss(all_preds_df['y_true'], all_preds_df['p_model'])
    acc = accuracy_score(all_preds_df['y_true'],
                         (all_preds_df['p_model'] > 0.5).astype(int))

    c_results.append({
        'C'          : C,
        'brier_score': round(bs, 4),
        'accuracy'   : round(acc, 4),
    })

c_df = pd.DataFrame(c_results).sort_values('brier_score')
print(c_df.to_string(index=False))

     C  brier_score  accuracy
 0.050       0.2248    0.6449
 0.100       0.2248    0.6431
 0.500       0.2248    0.6440
 1.000       0.2248    0.6440
 5.000       0.2248    0.6440
10.000       0.2248    0.6440
 0.010       0.2249    0.6449
 0.001       0.2287    0.6118


Este notebook entrena y evalúa el modelo predictivo usando tres funciones en secuencia.

---

### `build_features()` — construcción del dataset

Une el historial de ELO con el schedule limpio para construir los features
del modelo. Los ratings ELO son siempre **pre-partido** — el modelo nunca
ve información del futuro.

```
elo_history.csv  +  schedules limpio
        ↓
  unir por (season, week, home_team, away_team)
        ↓
  crear columnas:
    elo_diff  = elo_home_pre - elo_away_pre + HOME_ADVANTAGE
    week_norm = week / max_week_season  ← normaliza era 16 vs 17 juegos
        ↓
  filtrar solo REG + eliminar missings
  agregar sample_weight (2020 = 0.3, resto = 1.0)
        ↓
  DataFrame listo para modelar
```

> **Nota:** la estandarización (`StandardScaler`) ocurre dentro de
> `walk_forward_validation()`, no aquí. Eso evita que el scaler
> "vea" datos del futuro al calcular media y desviación estándar.

---

### `walk_forward_validation()` — evaluación del modelo

Valida el modelo respetando el orden temporal. No se puede usar K-Fold
estándar porque mezclaría partidos futuros con pasados (leakage).

En cada fold, el training window se expande para incluir la temporada anterior:

```
Fold 1:  Train 1999–2021  →  Test 2022  →  BS=0.2305  Acc=61.6%
Fold 2:  Train 1999–2022  →  Test 2023  →  BS=0.2338  Acc=61.4%
Fold 3:  Train 1999–2023  →  Test 2024  →  BS=0.2116  Acc=67.3%
Fold 4:  Train 1999–2024  →  Test 2025  →  BS=0.2232  Acc=67.3%
```

Visualmente:

```
1999 ──────────────────────────────────── 2025
|          TRAIN          | TEST |
|___________________2021__|_2022_|          Fold 1

|           TRAIN          | TEST |
|____________________2022__|_2023_|         Fold 2

|            TRAIN          | TEST |
|_____________________2023__|_2024_|        Fold 3

|             TRAIN          | TEST |
|______________________2024__|_2025_|       Fold 4
```

Dentro de cada fold el orden es estricto:

```python
# 1. Scaler fiteado SOLO sobre training → evita leakage
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # aprende media/std del training
X_test_sc  = scaler.transform(X_test)       # aplica esa escala al test

# 2. Entrenar sobre training
model.fit(X_train_sc, y_train, sample_weight=w_train)

# 3. Predecir sobre test
p_home = model.predict_proba(X_test_sc)[:, 1]
```

---

### `train_final_model()` — modelo para producción

Entrena el modelo final sobre **todos los datos disponibles** (1999–2025).
Este es el modelo que predice la temporada 2026.

```
walk_forward_validation()  →  evalúa el modelo (¿funciona el enfoque?)
                               usa splits train/test

train_final_model()        →  construye el modelo para producción
                               entrena con todos los datos disponibles
                               predice partidos 2026
```

La analogía:

```
walk_forward  →  examen de práctica
                 te dice qué tan bien funciona el enfoque

train_final   →  estudiar con todo el material antes del examen real
                 más datos = mejor modelo
                 la evaluación ya la hiciste en walk_forward
```

```
Train: 1999–2025  →  Coeficientes β fijos  →  Predice temporada 2026
```

---

### Flujo completo

```
build_features()
      ↓
  6,967 partidos REG con elo_diff, home_rest,
  away_rest, div_game, week_norm
      ↓
walk_forward_validation()
      ↓
  BS agregado = 0.2248  ·  Accuracy = 64.4%
  Hallazgo: elo_diff domina (β=0.679)
  Features adicionales aportan señal marginal
      ↓
train_final_model()
      ↓
  Modelo listo para predecir temporada 2026
  Coeficientes β fijos durante toda la temporada
  ELO se actualiza partido a partido
```

---

### Coeficientes β del modelo final

| Feature | Coeficiente β | Importancia relativa |
|---|---|---|
| `elo_diff` | 0.6790 | ██████████████████ 100% |
| `away_rest` | −0.0570 | ██ 8.4% |
| `div_game` | −0.0544 | ██ 8.0% |
| `week_norm` | 0.0332 | █ 4.9% |
| `home_rest` | 0.0153 | █ 2.3% |

El `elo_diff` domina con un coeficiente 12× mayor que cualquier otro feature.
Los features adicionales introducen ajustes marginales pero no mueven
significativamente el poder predictivo del modelo.

---

# Gradient Boosting

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from src.model import walk_forward_validation, FEATURES

TEST_SEASONS = [2022, 2023, 2024, 2025]

# ── Regresión logística ───────────────────────────────────────────────────────
print("=== Regresión Logística ===")
lr_results, lr_metrics = walk_forward_validation(
    features_df,
    test_seasons=TEST_SEASONS,
    model=LogisticRegression(C=1.0, max_iter=1000, random_state=42),
)

# ── Gradient Boosting ─────────────────────────────────────────────────────────
print("\n=== Gradient Boosting ===")
gb_results, gb_metrics = walk_forward_validation(
    features_df,
    test_seasons=TEST_SEASONS,
    model=GradientBoostingClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
    ),
)

# ── Tabla comparativa ─────────────────────────────────────────────────────────
from sklearn.metrics import brier_score_loss, accuracy_score

def resumen(nombre, results_df):
    y    = results_df['home_team_win']
    p    = results_df['p_model']
    bs   = brier_score_loss(y, p)
    acc  = accuracy_score(y, (p > 0.5).astype(int))
    return {'modelo': nombre, 'brier_score': round(bs, 4), 'accuracy': f"{acc:.1%}"}

comparacion = pd.DataFrame([
    {'modelo': 'Modelo nulo',         'brier_score': 0.2500, 'accuracy': '~56%'},
    {'modelo': 'ELO solo',            'brier_score': 0.2235, 'accuracy': '64.0%'},
    resumen('Regresión logística', lr_results),
    resumen('Gradient Boosting',   gb_results),
    {'modelo': 'Vegas (benchmark)',   'brier_score': 0.2114, 'accuracy': '~67%'},
])

print("\nTabla comparativa final:")
print(comparacion.to_string(index=False))

=== Regresión Logística ===
Modelo          : LogisticRegression
Scale features  : True
Features        : ['elo_diff', 'home_rest', 'away_rest', 'div_game', 'week_norm']
-----------------------------------------------------------------
Fold 2022 | Train=5,880 | Test=271 | BS=0.2305 | Acc=61.6%
Fold 2023 | Train=6,151 | Test=272 | BS=0.2338 | Acc=61.4%
Fold 2024 | Train=6,423 | Test=272 | BS=0.2116 | Acc=67.3%
Fold 2025 | Train=6,695 | Test=272 | BS=0.2232 | Acc=67.3%

=== Gradient Boosting ===
Modelo          : GradientBoostingClassifier
Scale features  : False
Features        : ['elo_diff', 'home_rest', 'away_rest', 'div_game', 'week_norm']
-----------------------------------------------------------------
Fold 2022 | Train=5,880 | Test=271 | BS=0.2312 | Acc=60.1%
Fold 2023 | Train=6,151 | Test=272 | BS=0.2359 | Acc=60.3%
Fold 2024 | Train=6,423 | Test=272 | BS=0.2118 | Acc=65.1%
Fold 2025 | Train=6,695 | Test=272 | BS=0.2213 | Acc=66.2%

Tabla comparativa final:
             modelo  b

In [9]:
from pathlib import Path
Path('../data/processed').mkdir(parents=True, exist_ok=True)

features_df.to_csv('../data/processed/model_features.csv', index=False)
coef_df.to_csv('../data/processed/model_coefficients.csv', index=False)

## § Resumen y conclusiones · Notebook 03

---

### Objetivo

Evaluar si agregar features adicionales al diferencial de ELO mejora
el poder predictivo del sistema. Se probaron tres enfoques en orden
de complejidad creciente.

---

### Features evaluados

| Feature | Descripción | Hipótesis |
|---|---|---|
| `elo_diff` | `elo_home_pre − elo_away_pre + HOME_ADVANTAGE` | Feature principal · diferencial de calidad entre equipos |
| `home_rest` | Días de descanso del local | Más descanso → ventaja del local |
| `away_rest` | Días de descanso del visitante | Más descanso → ventaja del visitante |
| `div_game` | 1 si es juego divisional | Rivalidades reducen ventaja del favorito |
| `week_norm` | `week / max_week_season` | Efecto de posición en la temporada |

> `week_norm` normaliza la transición de 16 a 17 juegos por temporada
> a partir de 2021. Sin normalizar, `week=16` significaría cosas
> distintas en eras distintas.

---

### Validación — walk-forward con ventana expandible

Se usó validación walk-forward para respetar el orden temporal de los datos.
K-Fold estándar no es apropiado para series de tiempo porque mezclaría
partidos futuros con pasados (leakage).

```
Fold 1:  Train 1999–2021  →  Test 2022
Fold 2:  Train 1999–2022  →  Test 2023
Fold 3:  Train 1999–2023  →  Test 2024
Fold 4:  Train 1999–2024  →  Test 2025
```

En cada fold el `StandardScaler` se fitea exclusivamente sobre el training set
para evitar que la escala del test contamine el modelo.

---

### Resultados por modelo

#### Regresión Logística

| Fold | Train | Test | Brier Score | Accuracy |
|---|---|---|---|---|
| 2022 | 5,880 | 271 | 0.2305 | 61.6% |
| 2023 | 6,151 | 272 | 0.2338 | 61.4% |
| 2024 | 6,423 | 272 | 0.2116 | 67.3% |
| 2025 | 6,695 | 272 | 0.2232 | 67.3% |
| **Agregado** | | **1,087** | **0.2248** | **64.4%** |

#### Gradient Boosting

| Fold | Train | Test | Brier Score | Accuracy |
|---|---|---|---|---|
| 2022 | 5,880 | 271 | 0.2312 | 60.1% |
| 2023 | 6,151 | 272 | 0.2359 | 60.3% |
| 2024 | 6,423 | 272 | 0.2118 | 65.1% |
| 2025 | 6,695 | 272 | 0.2213 | 66.2% |
| **Agregado** | | **1,087** | **0.2250** | **62.9%** |

---

### Tabla comparativa final

| Modelo | Brier Score | Accuracy | vs. ELO solo |
|---|---|---|---|
| Modelo nulo (50/50) | 0.2500 | ~56% | +0.0265 |
| **ELO solo** | **0.2235** | **64.0%** | **— referencia** |
| Regresión logística | 0.2248 | 64.4% | +0.0013 ← peor |
| Gradient Boosting | 0.2250 | 62.9% | +0.0015 ← peor |
| Vegas (benchmark) | 0.2114 | ~67% | −0.0121 |

---

### Coeficientes β del modelo logístico

| Feature | Coeficiente β | Importancia relativa |
|---|---|---|
| `elo_diff` | +0.6790 | ██████████████████ 100% |
| `away_rest` | −0.0570 | ██ 8.4% |
| `div_game` | −0.0544 | ██ 8.0% |
| `week_norm` | +0.0332 | █ 4.9% |
| `home_rest` | +0.0153 | █ 2.3% |

`elo_diff` domina con un coeficiente **12× mayor** que cualquier otro feature.

---

### Hallazgo principal

Ningún modelo superó al ELO puro en Brier Score.

La regresión logística y Gradient Boosting no mejoran el poder predictivo
porque los features adicionales (`home_rest`, `away_rest`, `div_game`,
`week_norm`) no contienen señal estadísticamente significativa más allá
de lo que ya captura el diferencial de ELO.

Esto se verificó con tres experimentos:

```
1. Coeficientes β          → features adicionales son 12–44× más pequeños que elo_diff
2. Solo elo_diff           → BS=0.2246 ≈ modelo completo (0.2248)
3. Tuning de C             → BS=0.2248 para cualquier C entre 0.05 y 10.0
```

El Gradient Boosting confirma que tampoco existen interacciones no lineales
relevantes entre los features disponibles.

---

### Interpretación

Este resultado **no es un fracaso** — es un hallazgo genuino sobre la
estructura del problema:

> *"El sistema ELO calibrado desde datos 1999–2025 captura prácticamente
> toda la señal predictiva disponible en las variables estructurales públicas
> de la NFL. La varianza partido a partido que no captura el ELO es
> inherente al deporte y no es recuperable con las variables disponibles."*

La información que le falta al sistema (lesiones en tiempo real, condiciones
climáticas exactas, rotaciones de plantilla) es precisamente lo que Vegas
incorpora y explica el gap restante de 0.0121 puntos de Brier Score.

---

### Decisión de diseño para Notebook 04

El motor Monte Carlo usará el **ELO puro** como función de predicción:

```python
P(home gana) = 1 / (1 + 10^(-(elo_home - elo_away + HOME_ADVANTAGE) / 400))
```

No se requiere cargar el modelo logístico ni el scaler en la simulación.
Los coeficientes β se documentan como referencia pero no se usan en producción.

---

### Outputs generados

| Archivo | Contenido | Usado en |
|---|---|---|
| `data/processed/model_features.csv` | Dataset con features construidos | Referencia / análisis |
| `data/processed/model_coefficients.csv` | Coeficientes β del modelo logístico | Referencia / documentación |